# MLIR-compiled ONNX SmolLM2 INT8 on PYNQ
Compile the upstream ONNX model using `docs/acceptance/smollm-onnx.md`. This demo uses only shared `src/` and independent model assets.

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ["NPU_REPO"]).resolve()
sys.path.insert(0, str(REPO))
PACKAGE = Path(os.environ["NPU_COMPILED_PACKAGE"]).resolve()
from src.inference.engine import LLMEngine
from pynq import Overlay

In [ ]:
overlay = Overlay(os.environ["NPU_OVERLAY"])
engine = LLMEngine(PACKAGE, overlay=overlay)
print(engine.meta["compiled_pipeline"])
print("Supported exact chat lengths:", engine.prefill_lengths)

## Edit the prompt
The complete chat token length must have a compiled prefill specialization. The default package supports 36, 40 and 42 tokens. Compile additional exact lengths for other prompts; generation stops at EOS or the context limit.

In [ ]:
prompt = "write a paragraph about npu"
ids, text, stats = engine.generate(
    prompt, max_new_tokens=32, chat=True,
    on_token=lambda _, piece: print(piece, end="", flush=True),
)
print()
print(stats)
print(engine.model.stats())

## Release board buffers
Run this when finished with the engine.

In [ ]:
for buffer in (engine.model.io, engine.model.stage, engine.model.words):
    if buffer is not None:
        buffer.buffer.freebuffer()
del engine